In [ ]:
import pandas as pd 
import re
from IPython.display import display, HTML

In [ ]:
even_sample = pd.read_parquet('../../data/salary_sample_body_benefits_v2.parquet.gzip')

In [ ]:
even_sample.rename(columns={'Has AI Skills':'AI ROLE'}, inplace=True)

In [ ]:
even_sample.to_parquet('../../data/salary_sample_body_benefits_v2.parquet.gzip', compression='gzip')

In [ ]:
benefits = ['CAREER_DEV', 'WLB', 'HEALTH_WELLBEING', 'RECOGNITION', 'FAMILY', 'CULTURE']

In [ ]:
career_dev = "training programs", "education assistance","tuition reimbursement", "tuition assistance", "mentorship", "career growth", "professional development", "leadership development", "career advancement", "leadership training", "growth opportunities", "personal development", "education reimbursement"
wlb = "paid time off", "PTO", "extra vacation", "flexible vacation", "mental health day", "work-life balance", "work/life balance","generous time-off", "generous time off", "paid vacation", "paid holidays", "holiday pay"
wellbeing = "wellness stipend", "mental health support", "gym membership", "wellness program", "well-being stipend", "wellness programs", "mental health benefits", "employee well-being"
health_wellbeing = "wellness stipend", "health benefits", "mental health support", "gym membership", "wellness program", "well-being stipend", "wellness programs", "mental health benefits", "employee well-being", "health care benefits"
recognition = "employee of the month", "long-term rewards", "recognition program", "peer recognition", "service awards", "employee recognition"
family = "parental leave", "childcare assistance", "childcare support", "childcare", "family support", "flexible maternity leave", "paternity leave", "maternity leave", "paid family leave", "child care discount"
culture = "collaborative environment", "diversity and inclusion", "team culture", "creative freedom", "values-driven", "inclusive culture", "diverse team", "inclusive environment"
keywords_list = [career_dev,wlb,health_wellbeing, recognition, family, culture]

# export keywords list as pickle
import pickle
with open('../../data/keywords_list.pkl', 'wb') as f:
    pickle.dump(keywords_list, f)

In [ ]:
# read in keywords_list.pkl
with open('../../data/keywords_list.pkl', 'rb') as f:
    keywords_list = pickle.load(f)


## Larger Sample 100 Per Benefit

In [ ]:
# # remove those already checked
# benefits = ['CAREER_DEV', 'WLB', 'WELLBEING', 'RECOGNITION', 'FAMILY', 'CULTURE']
# checked_ids = []
# for benefit, keywords in zip(benefits, keywords_list):
#     # print(benefit)
#     # print('--'*100)
#     benefit_df = even_sample[even_sample[benefit] == True]
#     test_sample = benefit_df.sample(25, random_state=2)
#     # checked_ids.append(test_sample['ID'])
#     for k, row in test_sample.iterrows():
#         i = row['BODY']
#         checked_ids.append(row['ID'])

        
# # remove those already checked


## Import Large Sample Checked

In [ ]:
checked_df = pd.read_excel('../../highlighted_job_postings.xlsx')

In [ ]:
checked_ids = checked_df['ID'].tolist()

In [ ]:
sample_notchecked = even_sample[~even_sample['ID'].isin(checked_ids)]


In [ ]:
def highlight_keywords_html(text, keywords):
    keywords = sorted(keywords, key=len, reverse=True)
    pattern = '|'.join(map(re.escape, keywords))
    found_keywords = re.findall(pattern, text, flags=re.IGNORECASE)

    # Replace keywords with bold version using HTML <b> tags
    highlighted_text = re.sub(r'(' + pattern + r')',r'<b><span style="background-color: yellow; color: black">\1</span></b>', text, flags=re.IGNORECASE)
    return highlighted_text, found_keywords

In [ ]:
pd.set_option('display.max_columns', None)

# Export Sample to Check

In [ ]:
checked_ids_2 = []
check_df = pd.DataFrame(columns=['Benefit', 'ID', 'AI Role', 'Description'])
html_output = []
new_rows = []

for benefit, keywords in zip(benefits, keywords_list):
    print(benefit)
    print('--'*100)
    benefit_df = sample_notchecked[sample_notchecked[benefit] == True]
    test_sample = benefit_df.sample(50, random_state=2)
    html_output.append(f'{benefit}</h3>')
    for k, row in test_sample.iterrows():
        i = row['BODY']
        checked_ids_2.append(row['ID'])
        highlighted_description_html, found_keywords = highlight_keywords_html(i, keywords)
        job_id_html = f'<h3>Job ID: {row["ID"]}</h3>'
        description_html = f'<p>{highlighted_description_html}</p>'
            
        # Combine and append to output
        html_output.append(job_id_html + description_html)
        print(f'AI Role: {row["AI ROLE"]}')
        display(HTML(highlighted_description_html))
        # add to check_df
        new_row = {
            'Benefit': benefit,
            'ID': row['ID'],
            'AI Role': row['AI ROLE'],
            'Description': highlighted_description_html,
            'Keywords Found': ', '.join(found_keywords) 
        }
        new_rows.append(new_row)
        
# Convert the list of new rows into a DataFrame
check_df = pd.DataFrame(new_rows)

# Optionally export to a CSV file
check_df.to_csv('../../exports/highlighted_job_postings_2.csv', index=False)
        
# export the highlighted_description_html HTML

final_html_content = '<html><body>' + '\n\n'.join(html_output) + '</body></html>'

with open('../../exports/highlighted_descriptions_2.html', 'w', encoding='utf-8') as f:
    f.write(final_html_content)

# Explode check_df

In [ ]:
# explode check_df on keywords found
check_df['Keywords Found'] = check_df['Keywords Found'].str.split(', ')
check_df = check_df.explode('Keywords Found')


In [ ]:
check_df

In [ ]:
check_df.to_csv('../../exports/highlighted_job_postings_2_exploded.csv', index=False)